# 🏆 CTF Agent Swarm — Google Colab Edition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

Selamat datang di **CTF Agent Swarm** versi Google Colab!
Framework ini adalah agen otonom berbasis multi-model AI yang mampu menganalisis file tantangan CTF (Reverse Engineering, Binary Exploitation/Pwn, Cryptography, Web Security, Forensics, dan Steganography), mengeksploitasi celah keamanan, dan menangkap flag secara otomatis.

---
### 💡 Fitur Utama & Opsi Tanpa Token / Gratis:
1. **Gratis Tanpa Biaya API**:
   - Menggunakan **Google Gemini API Free Tier** (dapatkan API Key gratis di [Google AI Studio](https://aistudio.google.com/)).
   - Menggunakan **Model Lokal / Ollama** (DeepSeek-R1 / Qwen2.5-Coder) memanfaatkan GPU T4 gratis dari Google Colab.
2. **Eksekusi Host / Local Sandbox**: Seluruh perintah dijalankan dalam workspace terisolasi dengan timeout pengaman.
3. **Deteksi Flag Otomatis**: Setiap pola flag (, ) langsung terdeteksi dari hasil analisis tool.
4. **Penyimpanan State SQLite**: Riwayat pengerjaan dan flag tersimpan otomatis di database lokal.

## ⚙️ Langkah 1: Persiapan Environment & Instalasi Dependensi
Jalankan sel di bawah ini untuk menginstal package manager cepat  dan seluruh dependensi CTF Agent.

In [ ]:
# 1. Cek informasi sistem dan GPU di Google Colab
!nvidia-smi || echo "Mode CPU aktif (tetap berjalan lancar!)"

# 2. Install Astral UV untuk instalasi paket super cepat
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

# 3. Clone repository atau pastikan direktori kerja siap
# !git clone https://github.com/username/CTFPROJECT.git /content/CTFPROJECT
# %cd /content/CTFPROJECT

# 4. Install dependensi Python pendukung CTF
!pip install --quiet pydantic-ai pydantic-settings rich click httpx pyyaml aiohttp beautifulsoup4 markdownify
!apt-get update -qq && apt-get install -y -qq binutils file xxd radare2 gdb strace steghide foremost tesseract-ocr > /dev/null
print("
[✓] Environment dan tools CTF berhasil disiapkan!")

## 🔑 Langkah 2: Konfigurasi API Key (Bisa Gratis!)
Pilih salah satu metode:
- **Metode Gratis (Rekomendasi)**: Dapatkan API Key gratis di [Google AI Studio](https://aistudio.google.com/app/apikey) tanpa kartu kredit.
- **Metode Berbayar/Opsional**: Masukkan OpenAI / Anthropic / DeepSeek API key jika memiliki token tersebut.
- **Metode Rahasia Colab**: Simpan di menu *Secrets* (ikon kunci di sebelah kiri) dengan nama .

In [ ]:
# Masukkan API Key Anda di bawah ini (atau gunakan Colab Secrets)
import os
from getpass import getpass

try:
    from google.colab import userdata
    gemini_key = userdata.get("GEMINI_API_KEY")
except Exception:
    gemini_key = None

if not gemini_key:
    print("Masukkan Google Gemini API Key (Gratis dari https://aistudio.google.com/):")
    gemini_key = getpass("GEMINI_API_KEY: ")

if gemini_key:
    os.environ["GEMINI_API_KEY"] = gemini_key
    print("[✓] GEMINI_API_KEY berhasil disetel!")
else:
    print("[!] Tidak ada API key yang disetel. Pastikan Anda menggunakan model lokal jika tanpa token.")

# Konfigurasi CTFd opsional (jika terhubung ke kompetisi langsung)
os.environ["CTFD_URL"] = "http://localhost:8000"  # Ganti dengan URL CTFd kompetisi
os.environ["CTFD_TOKEN"] = ""                    # Token CTFd Anda (jika ada)
os.environ["SANDBOX_MODE"] = "local"             # Mode lokal untuk Colab


## 📁 Langkah 3: Menyiapkan Tantangan / Soal CTF
Anda dapat membuat folder soal secara manual atau mendownload file tantangan.

In [ ]:
# Contoh membuat direktori soal latihan (misal: kategori crypto / reverse)
import os
from pathlib import Path

ch_dir = Path("challenges/crypto_caesar_warmup")
(ch_dir / "distfiles").mkdir(parents=True, exist_ok=True)

# 1. Simpan metadata soal
metadata_content = """name: "Caesar Warmup"
category: "crypto"
value: 100
description: "Kami mencegat ciphertext berikut: 'synt{pbyno_fbyire_vf_njrfbzr}'. Bisakah kamu memecahkannya?"
hints:
  - content: "Cobalah teknik rotasi klasik (ROT-13 / Caesar cipher)."
"""
(ch_dir / "metadata.yml").write_text(metadata_content)

# 2. Simpan distfile jika ada file lampiran
(ch_dir / "distfiles" / "cipher.txt").write_text("synt{pbyno_fbyire_vf_njrfbzr}
")

print(f"[✓] Soal simulasi berhasil dibuat di {ch_dir}")
!ls -la challenges/crypto_caesar_warmup

## 🚀 Langkah 4: Jalankan CTF Solver Otonom!
Jalankan CTF Agent untuk menganalisis dan memecahkan soal secara mandiri.

In [ ]:
# Menjalankan solver pada tantangan di atas
# Flag: --sandbox-mode local memastikan berjalan mulus di lingkungan Colab
!python -m backend.cli --challenge challenges/crypto_caesar_warmup --sandbox-mode local -v

## 📊 Langkah 5: Lihat Hasil & Laporan Kompetisi
Database SQLite menyimpan seluruh flag yang berhasil dipecahkan beserta riwayat submisi.

In [ ]:
# Tampilkan tabel rekapitulasi poin dan flag yang diperoleh
!python -m backend.cli --report

---
### 💡 Tips Menjalankan di Mode Swarm / Live Competition
Jika ingin menghubungkan agen ke platform CTFd sungguhan untuk memonitor dan menyelesaikan semua soal secara otomatis:
